project

In [1]:
# ==========================================
# COZY CRAFT COLLECTIVE ANALYTICS PIPELINE
# SQL → Python → Power BI
# ==========================================

import pandas as pd
import numpy as np
import psycopg2

# Connect to PostgreSQL
conn = psycopg2.connect(
    dbname="cozycraft",   
    user="postgres",
    password="postgre123",
    host="localhost",
    port="5432"
)

print("Connected")

Connected


In [22]:
# ==========================================
# LOAD DATA FROM DATA WAREHOUSE
# ==========================================

event_data = pd.read_sql("""
SELECT *
FROM fact_event_performance
""", conn)

attendees = pd.read_sql("""
SELECT *
FROM dim_attendees
""", conn)

event_data.head()

C:\Users\kevin\AppData\Local\Temp\ipykernel_21440\884019888.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  event_data = pd.read_sql("""
C:\Users\kevin\AppData\Local\Temp\ipykernel_21440\884019888.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  attendees = pd.read_sql("""


,event_name,event_start_date,registrations,checked_in,attendance_rate,venue_capacity
0,January Cozy Craft Meetup #2 - Lakewood,1/18/2026,76,27,0.36,80
1,November Cozy Craft Meetup,11/9/2025,55,16,0.29,80
2,Cozy Craft Meetup,10/12/2025,30,0,0.00,80
3,February BHM Cozy Craft Meetup,2/8/2026,92,49,0.53,80
4,December Cozy Craft Meetup,12/7/2025,57,0,0.00,80


In [23]:
# ==========================================
# DATA CLEANING
# ==========================================

# Fix date format
event_data["event_start_date"] = pd.to_datetime(event_data["event_start_date"])

# Rename for clarity
event_data = event_data.rename(columns={
    "event_start_date": "event_date"
})

# ==========================================
# DATA QUALITY LAYER
# ==========================================

# Flag reliable vs limited data
event_data["data_quality_flag"] = event_data["event_date"].dt.year.apply(
    lambda x: "Reliable" if x == 2026 else "Limited"
)

# Flag completed vs upcoming events
event_data["event_status"] = event_data["event_date"].apply(
    lambda x: "Completed" if x < pd.Timestamp.today() else "Upcoming"
)

# Preview cleaned data
event_data.sort_values("event_date")

,event_name,event_date,registrations,checked_in,attendance_rate,venue_capacity,data_quality_flag,event_status
2,Cozy Craft Meetup,2025-10-12,30,0,0.00,80,Limited,Completed
1,November Cozy Craft Meetup,2025-11-09,55,16,0.29,80,Limited,Completed
4,December Cozy Craft Meetup,2025-12-07,57,0,0.00,80,Limited,Completed
6,January Cozy Craft Meetup #1,2026-01-04,82,39,0.48,80,Reliable,Completed
0,January Cozy Craft Meetup #2 - Lakewood,2026-01-18,76,27,0.36,80,Reliable,Completed
3,February BHM Cozy Craft Meetup,2026-02-08,92,49,0.53,80,Reliable,Completed
7,March Cozy Craft Meetup,2026-03-08,119,54,0.45,80,Reliable,Completed
5,Spring Fling Cozy Craft Meetup,2026-04-12,98,0,0.00,80,Reliable,Upcoming


In [24]:
# ==========================================
# RELIABLE BASELINE EVENTS (2026 ONLY)
# ==========================================

valid_events = event_data[
    (event_data["data_quality_flag"] == "Reliable") &
    (event_data["event_status"] == "Completed")
]

In [25]:
# ==========================================
# CORE KPI CALCULATIONS
# ==========================================

avg_show_rate = (
    valid_events["checked_in"].sum() /
    valid_events["registrations"].sum()
)

no_show_rate = 1 - avg_show_rate

capacity = 80

In [26]:
# ==========================================
# CURRENT EVENT ANALYSIS
# ==========================================

upcoming_event = event_data[event_data["event_status"] == "Upcoming"]

registrations = upcoming_event["registrations"].values[0]

# Attendance if event happened today
current_attendance_estimate = registrations * avg_show_rate

In [27]:
# ==========================================
# REGISTRATION GROWTH
# ==========================================

event_sorted = event_data.sort_values("event_date")

event_sorted["growth"] = event_sorted["registrations"].pct_change()

avg_registration_growth = event_sorted["growth"].mean()

In [28]:
# ==========================================
# FORECAST FUTURE EVENTS
# ==========================================

future_events = pd.DataFrame({
    "future_event_number": range(1, 6)
})

future_events["predicted_registrations"] = [
    int(registrations * ((1 + avg_registration_growth) ** i))
    for i in future_events["future_event_number"]
]

future_events["predicted_attendance"] = (
    future_events["predicted_registrations"] * avg_show_rate
).round()

future_events

,future_event_number,predicted_registrations,predicted_attendance
0,1,119,55.0
1,2,146,67.0
2,3,179,82.0
3,4,219,100.0
4,5,268,123.0


In [29]:
# ==========================================
# CAPACITY ANALYSIS
# ==========================================

registrations_needed = capacity / avg_show_rate

In [30]:
# ==========================================
# LOYALTY SEGMENTATION
# ==========================================

def segment(events):
    if events == 1:
        return "1 Event (First-Time)"
    elif events == 2:
        return "2 Events (Returning)"
    elif 3 <= events <= 4:
        return "3-4 Events (Engaged)"
    else:
        return "5+ Events (Core)"

attendees["loyalty_segment"] = attendees["events_attended"].apply(segment)

In [41]:
# ==========================================
# SECTION 11 — REBUILD LOYALTY SEGMENTS (CLEAN)
# ==========================================

def segment_user(events):
    if events == 1:
        return "1 Event (First-Time)"
    elif events == 2:
        return "2 Events (Returning)"
    elif events in [3, 4]:
        return "3-4 Events (Engaged)"
    elif events >= 5:
        return "5+ Events (Core)"

attendees["loyalty_segment"] = attendees["events_attended"].apply(segment_user)

In [42]:
loyalty_distribution = (
    attendees["loyalty_segment"]
    .value_counts()
    .reset_index()
)

loyalty_distribution.columns = ["segment", "count"]

# ORDER PROPERLY
order = [
    "1 Event (First-Time)",
    "2 Events (Returning)",
    "3-4 Events (Engaged)",
    "5+ Events (Core)"
]

loyalty_distribution["segment"] = pd.Categorical(
    loyalty_distribution["segment"],
    categories=order,
    ordered=True
)

loyalty_distribution = loyalty_distribution.sort_values("segment")

loyalty_distribution

,segment,count
0,1 Event (First-Time),277
1,2 Events (Returning),43
2,3-4 Events (Engaged),20
3,5+ Events (Core),7


In [43]:
loyalty_distribution["count"].sum(), len(attendees)

(347, 348)

In [44]:
attendees[attendees["events_attended"] == 0]

,attendee_email,events_attended,loyalty_level,loyalty_segment
349,None,0,Occasional,None


In [45]:
attendees = attendees[attendees["events_attended"] > 0]

In [46]:
attendees["loyalty_segment"] = attendees["events_attended"].apply(segment_user)

In [47]:
loyalty_distribution = (
    attendees["loyalty_segment"]
    .value_counts()
    .reset_index()
)

loyalty_distribution.columns = ["segment", "count"]

In [48]:
loyalty_distribution["count"].sum(), len(attendees)

(347, 347)

In [53]:
# ==========================================
# LOYALTY DISTRIBUTION
# ==========================================

loyalty_distribution = attendees["loyalty_segment"].value_counts().reset_index()
loyalty_distribution.columns = ["segment", "count"]

loyalty_distribution

,segment,count
0,1 Event (First-Time),277
1,2 Events (Returning),43
2,3-4 Events (Engaged),20
3,5+ Events (Core),7


In [54]:
# ==========================================
# EVENTS ATTENDED DISTRIBUTION
# ==========================================

events_attended_distribution = attendees["events_attended"].value_counts().sort_index().reset_index()
events_attended_distribution.columns = ["events_attended", "count"]

# Remove invalid 0
events_attended_distribution = events_attended_distribution[
    events_attended_distribution["events_attended"] > 0
]

events_attended_distribution

,events_attended,count
0,1,277
1,2,43
2,3,15
3,4,5
4,5,3
5,6,1
6,7,1
7,8,2


In [55]:
# ==========================================
# COMMUNITY METRICS
# ==========================================

total_attendees = attendees.shape[0]
repeat_attendees = attendees[attendees["events_attended"] > 1].shape[0]
retention_rate = repeat_attendees / total_attendees

core_members = attendees[attendees["events_attended"] >= 5].shape[0]
engaged_members = attendees[(attendees["events_attended"] >= 3) & (attendees["events_attended"] <= 4)].shape[0]
returning_members = attendees[attendees["events_attended"] == 2].shape[0]
new_members = attendees[attendees["events_attended"] == 1].shape[0]

In [58]:
# ==========================================
# FUTURE REGISTRATION FORECAST
# ==========================================

future_events = pd.DataFrame({
    "event_number": range(1, 6)
})

future_events["predicted_registrations"] = [
    round(registrations * ((1 + avg_registration_growth) ** i))
    for i in future_events["event_number"]
]

future_events

,event_number,predicted_registrations
0,1,120
1,2,147
2,3,179
3,4,219
4,5,268


In [59]:
# Apply show rate to forecast attendance
future_events["predicted_attendance"] = (
    future_events["predicted_registrations"] * avg_show_rate
).round()

In [60]:
# ==========================================
# WHEN DO WE HIT CAPACITY?
# ==========================================

capacity_event = future_events[
    future_events["predicted_registrations"] >= capacity
].head(1)

attendance_capacity_event = future_events[
    future_events["predicted_attendance"] >= capacity
].head(1)

capacity_event, attendance_capacity_event

(   event_number  predicted_registrations  predicted_attendance
 0             1                      120                  55.0,
    event_number  predicted_registrations  predicted_attendance
 2             3                      179                  82.0)

In [61]:
def format_event_label(df):
    if len(df) > 0:
        return f"Event +{int(df['event_number'].values[0])}"
    else:
        return "Beyond Forecast"

capacity_label = format_event_label(capacity_event)
attendance_capacity_label = format_event_label(attendance_capacity_event)

capacity_label, attendance_capacity_label

('Event +1', 'Event +3')

In [66]:
# ==========================================
# FINAL KPI + BUSINESS INSIGHTS SECTION (LOCKED)
# ==========================================

# --- OVERBOOKING INSIGHT ---
overbooking_ratio = registrations_needed / capacity

# --- CONVERSION INSIGHT ---
conversion_rate = returning_members / new_members

# --- ATTENDANCE GAP (NEW) ---
attendance_gap = capacity - round(current_attendance_estimate)


# ==========================================
# FINAL PRINT SUMMARY
# ==========================================

print("\n" + "="*70)
print("COZY CRAFT COLLECTIVE – FINAL ANALYTICAL SUMMARY")
print("="*70)

print("\n--- DATA QUALITY ---")
print(f"Reliable Events Used (2026): {len(valid_events)}")

print("\n--- ATTENDANCE BASELINE ---")
print(f"Average Show Rate: {avg_show_rate:.2%}")
print(f"No-Show Rate: {no_show_rate:.2%}")

print("\n--- CURRENT EVENT ---")
print(f"Current Registrations: {registrations}")
print(f"Estimated Attendance (Today): {round(current_attendance_estimate)}")
print(f"Attendance Gap (Seats Empty): {attendance_gap}")

print("\n--- CAPACITY INSIGHT ---")
print(f"Venue Capacity: {capacity}")
print(f"Registrations Needed for Full Capacity: {round(registrations_needed)}")
print(f"Overbooking Ratio Needed: {overbooking_ratio:.2f}x")
print(f"Capacity Reached (Registrations): {capacity_label}")
print(f"Capacity Reached (Actual Attendance): {attendance_capacity_label}")

print("\n--- KEY INSIGHT ---")
print("Low show rates (~46%) require significant overbooking (~2.2x) to consistently reach full venue capacity.")

print("\n--- GROWTH ---")
print(f"Average Registration Growth Per Event: {avg_registration_growth:.2%}")

print("\n--- COMMUNITY ---")
print(f"Total Attendees: {total_attendees}")
print(f"Repeat Attendees: {repeat_attendees}")
print(f"Retention Rate: {retention_rate:.2%}")
print(f"1 → 2 Event Conversion Rate: {conversion_rate:.2%}")

print("\n--- LOYALTY BREAKDOWN ---")
print(f"Core Members (5+): {core_members}")
print(f"Engaged Members (3-4): {engaged_members}")
print(f"Returning Members (2): {returning_members}")
print(f"First-Time Attendees (1): {new_members}")

print("\n" + "="*70)


# ==========================================
# FINAL METRICS TABLE (POWER BI READY)
# ==========================================

final_metrics = pd.DataFrame({
    "Metric":[
        "Reliable Events (2026)",
        "Average Show Rate",
        "No-Show Rate",
        "Venue Capacity",
        "Current Registrations",
        "Estimated Attendance (Today)",
        "Attendance Gap",
        "Predicted Attendance (Next Event)",
        "Registrations Needed",
        "Overbooking Ratio",
        "Capacity (Registration-Based)",
        "Capacity (Attendance-Based)",
        "Avg Growth Rate",
        "Total Attendees",
        "Repeat Attendees",
        "Retention Rate",
        "Conversion Rate (1→2)",
        "Core Members",
        "Engaged Members",
        "Returning Members",
        "First-Time Attendees"
    ],
    "Value":[
        len(valid_events),
        round(avg_show_rate,4),
        round(no_show_rate,4),
        capacity,
        registrations,
        round(current_attendance_estimate),
        attendance_gap,
        round(future_events.iloc[0]["predicted_attendance"]),
        round(registrations_needed),
        round(overbooking_ratio,2),
        capacity_label,
        attendance_capacity_label,
        round(avg_registration_growth,4),
        total_attendees,
        repeat_attendees,
        round(retention_rate,4),
        round(conversion_rate,4),
        core_members,
        engaged_members,
        returning_members,
        new_members
    ]
})

final_metrics


COZY CRAFT COLLECTIVE – FINAL ANALYTICAL SUMMARY

--- DATA QUALITY ---
Reliable Events Used (2026): 4

--- ATTENDANCE BASELINE ---
Average Show Rate: 45.80%
No-Show Rate: 54.20%

--- CURRENT EVENT ---
Current Registrations: 98
Estimated Attendance (Today): 45
Attendance Gap (Seats Empty): 35

--- CAPACITY INSIGHT ---
Venue Capacity: 80
Registrations Needed for Full Capacity: 175
Overbooking Ratio Needed: 2.18x
Capacity Reached (Registrations): Event +1
Capacity Reached (Actual Attendance): Event +3

--- KEY INSIGHT ---
Low show rates (~46%) require significant overbooking (~2.2x) to consistently reach full venue capacity.

--- GROWTH ---
Average Registration Growth Per Event: 22.32%

--- COMMUNITY ---
Total Attendees: 347
Repeat Attendees: 70
Retention Rate: 20.17%
1 → 2 Event Conversion Rate: 15.52%

--- LOYALTY BREAKDOWN ---
Core Members (5+): 7
Engaged Members (3-4): 20
Returning Members (2): 43
First-Time Attendees (1): 277



,Metric,Value
0,Reliable Events (2026),4
1,Average Show Rate,0.458
2,No-Show Rate,0.542
3,Venue Capacity,80
4,Current Registrations,98
5,Estimated Attendance (Today),45
6,Attendance Gap,35
7,Predicted Attendance (Next Event),55
8,Registrations Needed,175
9,Overbooking Ratio,2.18


In [74]:
# ==========================================
# EXPORT CLEAN DATA FOR POWER BI
# ==========================================

base_path = r"C:\Denver cozy crafts collevtive\\"

final_metrics.to_csv(base_path + "final_metrics.csv", index=False)
future_events.to_csv(base_path + "future_events.csv", index=False)
loyalty_distribution.to_csv(base_path + "loyalty_distribution.csv", index=False)
events_attended_distribution.to_csv(base_path + "events_attended_distribution.csv", index=False)
event_data.to_csv(base_path + "event_data_clean.csv", index=False)

print("Export complete.")

Export complete.
